# Hindi Text Restoration and Translation - end-to-end Colab notebook

This notebook is a guided walkthrough of a complete NLP project. You do not
need any prior NLP experience. Read the markdown above each code cell, run the
cell, and compare what you see with the explanation.

The setup cell **clones the public repository automatically**, so you can open
this notebook from GitHub and press **Runtime > Run all** without uploading a
folder first.

## What problem are we solving?

The input is Hindi text that has lost its punctuation, and its sentences are in
the wrong order. We must:

1. put punctuation and sentence boundaries back,
2. recover a sensible sentence order,
3. translate the ordered text into English,
4. keep every five sentences together as a group.

## What you will build

A three-stage pipeline:

```text
unpunctuated Hindi words
        |
        v
[1] punctuation model  ->  Hindi sentences
        |
        v
[2] ordering model     ->  sentences in a better order
        |
        v
[3] translation model  ->  English sentences and groups
```

The models are **mBERT** for stages 1 and 2 and **IndicTrans2** for stage 3.
No LLM or chatbot API is used anywhere.

## How to use this notebook

- Run the cells **from top to bottom** or use **Runtime > Run all**.
- Keep the GPU runtime on. Training on a CPU would be very slow.
- The first install and the training steps take time. That is normal.
- Do not skip the self-check cells: they tell you when something is wrong.
- You can read the output without understanding every line of code.

**Expected total time on a free Colab T4:** roughly 30-60 minutes, depending on
download speed and GPU availability.

## What each stage needs

| Stage | Input | Model | Output |
| --- | --- | --- | --- |
| Punctuation | words | mBERT + LoRA | one punctuation label per word |
| Ordering | sentences | mBERT + LoRA | one order for all sentences |
| Translation | Hindi sentences | IndicTrans2 | English sentences |


## The big picture, in plain English

Think of the pipeline as three people working on the same article:

1. **The editor** reads unpunctuated words and decides where commas and full
   stops belong. This also tells us where one sentence ends and the next begins.
2. **The librarian** receives the sentences in a random order and decides which
   sentence probably came first, second, third, and so on.
3. **The translator** turns each Hindi sentence into English, keeping the
   librarian's order.

### Important words you will see

| Word | Simple meaning |
| --- | --- |
| **NLP** | Natural Language Processing: teaching computers to work with human language. |
| **Model** | A program with adjustable numbers (weights) that learns patterns from examples. |
| **Training** | Showing a model many examples so it can learn those patterns. |
| **Inference** | Using a trained model to make predictions on new text. |
| **Token** | A piece of text the model reads. A Hindi word may be split into several tokens. |
| **Label** | The answer we want the model to predict, such as `comma` or `sentence_end`. |
| **Adapter** | A small saved set of trained weights. It lets us reuse a big model cheaply. |
| **GPU** | A fast parallel chip used for deep learning. Colab gives us one for free. |

You will see these terms again in the glossary at the end.


## Before you start

Check these boxes:

- [ ] You opened this notebook in Google Colab.
- [ ] **Runtime > Change runtime type > T4 GPU** is selected.
- [ ] You have a stable internet connection.
- [ ] Optional, for the final translation stage: you accepted the IndicTrans2
      licence on Hugging Face and added an `HF_TOKEN` secret in Colab.

The next code cell clones the public GitHub repository if it is not already
present. You do **not** need to upload the project folder manually.

If the GPU check below fails, stop and change the runtime type before
continuing.


## 0. Find or clone the project

Colab starts in `/content`. The next cell looks for the project, clones the
public GitHub repository if needed, and changes into the project root.

**Why this matters:** every later command uses relative paths such as
`src/`, `data/raw/`, and `scripts/`. Running from the wrong folder is the most
common reason a cell fails.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

PROJECT_DIR = Path("/content/nlp-zero-to-hero")
REPO_URL = "https://github.com/Vaibhavvv2524/nlp-zero-to-hero.git"

cwd = Path.cwd().resolve()
if (cwd / "src").is_dir():
    root = cwd
elif (cwd.parent / "src").is_dir():
    root = cwd.parent
elif (PROJECT_DIR / "src").is_dir():
    root = PROJECT_DIR
    if (root / ".git").is_dir():
        print(f"Updating {root}")
        subprocess.run(
            ["git", "-C", str(root), "pull", "--ff-only"],
            check=False,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
else:
    root = PROJECT_DIR
    if root.exists():
        print(f"Removing incomplete folder at {root}")
        shutil.rmtree(root)
    print(f"Cloning {REPO_URL} into {root}")
    root.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(root)],
        check=True,
    )

os.chdir(root)
sys.path.insert(0, str(root))

required = ["src", "data/raw", "requirements-colab.txt", "scripts/colab_setup.sh"]
missing = [name for name in required if not (root / name).exists()]
assert not missing, f"Missing project files: {missing}"

print("Working directory:", os.getcwd())
print("Files:", sorted(os.listdir("."))[:12])


### What you should see

The working directory should be `/content/nlp-zero-to-hero`, and the first few
file names should include `README.md`, `data`, `models`, `notebooks`, `src`,
and `scripts`.

If you see `Missing project files`, upload or clone the project again. The
folder must contain `src/`, `data/raw/`, `requirements-colab.txt` and
`scripts/colab_setup.sh`.


## 0.1 Check the GPU

Deep learning uses a **GPU** instead of a CPU because a GPU can do thousands of
small calculations at the same time. Colab's free T4 GPU has about 16 GB of
memory. Our pipeline deliberately caps each stage at 1.5 GB.

`torch.cuda.is_available()` answers: "can PyTorch see an NVIDIA GPU?"


In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "No GPU detected. In Colab use Runtime > Change runtime type > T4 GPU."
)

props = torch.cuda.get_device_properties(0)
print("GPU:", props.name)
print("Total GPU memory (GB):", round(props.total_memory / 1024**3, 2))


### If the GPU check fails

Go to **Runtime > Change runtime type**, choose **T4 GPU**, and run the
notebook again from the top. If Colab says no GPU is available, wait a few
minutes and try again; free GPUs are shared.


## 1. Install the Python packages

A **package** is reusable code written by other people. We need:

| Package | What it does here |
| --- | --- |
| `transformers` | Loads mBERT and IndicTrans2 and runs their models. |
| `peft` | Adds LoRA, a cheap way to fine-tune a large model. |
| `datasets` | Streams Hindi Wikipedia and the IITB translation test set. |
| `sacrebleu` | Computes BLEU and chrF translation scores. |
| `IndicTransToolkit` | Prepares text specifically for IndicTrans2. |
| `accelerate` | Helps the Hugging Face training loop run efficiently. |

We use **pinned versions** in `requirements-colab.txt`. Newer versions of some
packages removed features that IndicTrans2 still needs, so "just install the
latest" can break the project.

This cell downloads and installs everything. It can take several minutes.


In [ ]:
!bash scripts/colab_setup.sh


### Check that the install worked

If these imports succeed, the main packages are available. If Colab asks you to
restart the runtime after installing, do that, then run the notebook from the
top again (the install cell is safe to re-run).


In [ ]:
import datasets
import peft
import sacrebleu
import torch
import transformers

print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("peft        ", peft.__version__)
print("datasets    ", datasets.__version__)
print("sacrebleu   ", sacrebleu.__version__)
print("GPU         ", torch.cuda.get_device_name(0))


### 1.1 Optional: give Colab access to IndicTrans2

Training stages 1 and 2 use the public `bert-base-multilingual-cased` model and
need no login.

Stage 3 and the translation evaluation use the **gated** IndicTrans2 checkpoint.
If you have accepted its licence on Hugging Face, add a Colab secret named
`HF_TOKEN` (key icon in the left sidebar), then run the next cell. If you do not
have a token, you can still run every training cell; evaluation and the final
pipeline will stop with a clear authentication error.


In [ ]:
import os

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")

if token:
    from huggingface_hub import login
    login(token=token, add_to_git_credential=False)
    print("Hugging Face login: OK")
else:
    print("No HF_TOKEN found.")
    print("Training can continue. Evaluation/translation need IndicTrans2 access.")
    print("Optional interactive login:")
    print("  from huggingface_hub import notebook_login; notebook_login()")


## 2. Import project settings, fix the seed, and cap GPU memory

Three ideas in this cell:

1. **Shared settings.** `src/constants.py` stores paths, model names, labels and
   hyper-parameters in one place. Every stage reads from it, so the project has
   one source of truth.
2. **Seed.** `set_seed(42)` makes random choices repeatable. Training still has
   some GPU nondeterminism, but the data split and shuffling are reproducible.
3. **Memory cap.** `limit_gpu_memory(1.5)` tells PyTorch to use at most 1.5 GB
   of GPU memory for this process. If a stage exceeds the budget, it fails
   loudly instead of quietly using more. This is an assignment requirement.

The next cell also prints the five punctuation labels so you can see exactly
what stage 1 predicts.


In [ ]:
import json
import time
from pathlib import Path

from src.constants import (
    BASE_MODEL,
    GROUP_SIZE,
    LABELS,
    MEMORY_LIMIT_GB,
    ORDER_DATA,
    PUNCT_DATA,
    PUNCT_MODEL_DIR,
    ORDER_MODEL_DIR,
    PUNCTUATION,
    SEED,
    TRANSLATION_MODEL,
)
from src.utils import gpu_memory, limit_gpu_memory, read_jsonl, set_seed

set_seed(SEED)

print("Memory cap:", limit_gpu_memory(MEMORY_LIMIT_GB))
print("Memory now:", gpu_memory())
print()
print("Punctuation labels:")
for index, label in enumerate(LABELS):
    print(f"  {index}: {label:14s} -> {PUNCTUATION[label]!r}")
print()
print("Base model:     ", BASE_MODEL)
print("Translator:     ", TRANSLATION_MODEL)
print("Group size:     ", GROUP_SIZE)
print("Ordering strategy: borda (the default)")


## 3. Data: where do training examples come from?

A model learns from **examples**, not from rules we write by hand. Each training
example has:

- an **input** (what the model sees), and
- a **target** (the answer we want).

For punctuation, one example is a window of Hindi words. The target is one
label per word.

For ordering, one example is two sentences. The target is `1` if the first
sentence really comes before the second, and `0` otherwise.

We do **not** have a ready-made dataset for this exact assignment, so we build
one from Hindi Wikipedia:

- Wikipedia sentences already contain punctuation. We can remove the punctuation
  to create the input, and keep the original marks as the labels.
- Consecutive Wikipedia sentences give us correct order. Swapping them gives us
  incorrect order.

### Why Hindi needs care

Hindi uses the **danda** `।` as a full stop, and words are separated by spaces.
A naive regex like `\w` treats Devanagari vowel signs as separators and breaks
words (`अयोध्या` can become `अयधय`). The project therefore uses `str.split()`
and a small punctuation-aware helper instead.


### 3.1 Look at one raw assignment file

`data/raw/sample-1.txt` is a bag of words:

- words are separated by spaces,
- there is no punctuation,
- the special token `NULL` may appear and must be removed,
- sentence order is shuffled.

Run the cell and notice that there are no sentence-ending marks.


In [ ]:
from pathlib import Path

raw_path = Path("data/raw/sample-1.txt")
raw_text = raw_path.read_text(encoding="utf-8")
raw_tokens = raw_text.split()

print("File:", raw_path)
print("Tokens:", len(raw_tokens))
print("NULL tokens:", raw_tokens.count("NULL"))
print()
print("First 40 tokens:")
print(" ".join(raw_tokens[:40]))


### 3.2 What is a label?

A label is the answer for one word. Stage 1 uses five labels:

| Label | Mark added after the word | Meaning |
| --- | --- | --- |
| `none` | nothing | the word is inside a sentence |
| `comma` | `,` | a comma follows |
| `sentence_end` | `।` | the sentence ends here |
| `question` | `?` | a question ends here |
| `exclamation` | `!` | an exclamation ends here |

A window of 64 words might contain 58 `none` labels and only a few punctuation
labels. That imbalance is normal for language, and we handle it with
**class weights** during training.


### 3.3 How the dataset is built

1. Stream Hindi Wikipedia articles.
2. Split each article into sentences using `।`, `॥`, `?` and `!` (the label helper also recognises a final `.`).
3. Slide a window of 32-96 words over the article.
4. Shuffle the sentences **inside** the window. This mimics the assignment
   input and stops the model from using cross-sentence continuity as a shortcut.
5. Store the words and their original labels.
6. For ordering, create consecutive pairs `(A, B, 1)` and swapped pairs
   `(B, A, 0)`.

Run data preparation now. It downloads a stream of Wikipedia and writes JSONL
files under `data/processed/`.


In [ ]:
# --num-articles: how many usable Wikipedia articles to keep.
# --max-train-pairs: cap on ordering training pairs.
!python -m src.data_prep --num-articles 3000 --max-train-pairs 40000


### 3.4 Read the dataset summary

The next cell prints the summary and shows one real punctuation example and one
real ordering pair. Look for:

- **articles:** 3000 (the number we asked for),
- **splits:** 2700 train / 150 validation / 150 test,
- **punctuation train examples:** 25000 windows,
- **ordering train pairs:** 40000 pairs.

**Why three splits?**

- **Train:** the model learns from these examples.
- **Validation:** used during training to choose the best epoch.
- **Test:** never used for learning; only for the final honest score.


In [ ]:
summary = json.loads(Path("data/processed/data_prep_summary.json").read_text())
print("Articles:", summary["articles"])
print("Splits:", summary["splits"])
print()
print("Punctuation examples:")
for split, stats in summary["punctuation"].items():
    labels = ", ".join(f"{k}={v['count']}" for k, v in stats["labels"].items())
    print(f"  {split}: {stats['examples']} windows | {labels}")
print()
print("Ordering pairs:", {k: v["pairs"] for k, v in summary["ordering"].items()})

print("\nOne punctuation window (first 12 words):")
punct_example = read_jsonl(PUNCT_DATA / "test.jsonl")[0]
for word, label in list(zip(punct_example["words"], punct_example["labels"]))[:12]:
    print(f"  {word:18s} {label}")

print("\nOne ordering pair:")
order_example = read_jsonl(ORDER_DATA / "test.jsonl")[0]
print("  A:", order_example["sentence_a"][:100])
print("  B:", order_example["sentence_b"][:100])
print("  label:", order_example["label"], "(1 means A comes before B)")


### 3.5 Check the data before training

Training can waste a lot of GPU time if the data is wrong. The verifier checks
that words and labels align, window sizes are valid, pairs are balanced, and no
article appears in both train and test.

This is a **unit test for the dataset**. It should pass quickly.


In [ ]:
!python scripts/verify_colab_run.py --skip-models --skip-reports --skip-outputs


## 4. Stage 1: teach mBERT to restore punctuation

### What is mBERT?

**BERT** is a Transformer model: it reads a whole sequence of tokens and builds
a context-aware vector for each token. **mBERT** is the multilingual version,
trained on many languages including Hindi.

We use `bert-base-multilingual-cased`, which is about 178 million parameters.
Training all of them would need much more memory than 1.5 GB. Instead we use
**LoRA**.

### What is LoRA?

LoRA freezes the big pretrained model and adds small trainable matrices to a few
layers. Only those small matrices and the classification head are updated. The
result is an **adapter**: a tiny file that can be loaded on top of the base
model later.

Benefits:

- much less GPU memory,
- much faster training,
- the downloaded base model is reused across stages.

### How the classifier works

For every word, the model produces five scores: one for each punctuation label.
A **softmax** turns the scores into probabilities. The label with the highest
probability is the prediction.

The tricky part is **subwords**. mBERT may split one Hindi word into several
tokens. We put the word's label on the first subword and set the others to
`-100`, which the loss function ignores. This means each word is supervised
exactly once.


### 4.1 See how a Hindi word becomes subwords

This cell loads only the tokenizer (not the full model), so it is quick. The
`word_ids()` list shows which original word each token came from. Tokens with
`None` are special tokens such as `[CLS]` and `[SEP]`.

You do not need to memorise this. The goal is to see why label alignment needs
care.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
words = ["भारत", "एक", "विशाल", "देश", "है"]
encoding = tokenizer(words, is_split_into_words=True)

print("Original words:", words)
print("Subword tokens:", tokenizer.convert_ids_to_tokens(encoding["input_ids"]))
print("word_ids:      ", encoding.word_ids())


### 4.2 Train the punctuation adapter

The command below trains stage 1:

- `--epochs 3`: make three passes over the training data.
- `--batch-size 8`: look at 8 windows at a time. Smaller batches use less memory.
- `--learning-rate 3e-4`: how far the weights move after each batch.
- `--memory-gb 1.5`: enforce the assignment memory cap.

Training prints a table with the loss and the validation boundary F1. The loss
should generally go down. Boundary F1 is the score to watch: it measures how
well the model finds commas, sentence ends, question marks and exclamation
marks.


In [ ]:
# Stage 1: punctuation restoration.
!python -m src.punctuation.train \
    --epochs 3 \
    --batch-size 8 \
    --learning-rate 3e-4 \
    --memory-gb 1.5


### 4.3 Read the punctuation results

`training_summary.json` stores the final validation and test scores.

- **accuracy:** fraction of words whose label is correct. This is high even for
  a mediocre model because most words are `none`.
- **boundary F1:** the more important score. It treats "is this a sentence
  boundary?" as a yes/no decision and balances precision and recall.
- **per-class F1:** one score per label. `question` and `exclamation` are rare,
  so their F1 can be low or even `0.0` without meaning the whole model is
  broken.

A healthy full-data run should be clearly above chance. The verified local run
reached **0.8724 accuracy** and **0.8842 boundary F1** with fewer training
samples.


In [ ]:
punct_summary = json.loads((PUNCT_MODEL_DIR / "training_summary.json").read_text())
print("Trainable parameters:", punct_summary.get("trainable_parameters"))
print("Validation:", json.dumps(punct_summary["validation_metrics"], indent=2))
print("Test:", json.dumps(punct_summary["test_metrics"], indent=2))
print("Peak memory:", punct_summary["memory_at_end"])


### If stage 1 looks bad

- Check that data preparation finished and the verifier passed.
- A validation boundary F1 near `0.0` usually means the labels or adapter are
  missing, not that the model needs more epochs.
- If you run out of memory, restart the runtime, keep `--memory-gb 1.5`, and
  reduce `--batch-size` to 4.
- The notebook's self-check later enforces a minimum boundary F1.


## 5. Stage 2: teach a model which sentence comes first

Punctuation gives us sentences. Now we need their original order.

### Why not predict the whole order at once?

If an article has 20 sentences, there are `20!` possible orders. That is more
than 2 quintillion. Instead, we break the problem into a simpler question:

> Given sentence A and sentence B, which one comes first?

This is a **pairwise classifier**. It is trained on:

- `(A, B) -> 1` when A really comes before B,
- `(B, A) -> 0` for the swapped pair.

The model reads both sentences together as `[CLS] A [SEP] B [SEP]` and outputs
two scores: "B before A" and "A before B". A softmax turns them into
probabilities.

At inference we score many pairs, then use a **decoder** to turn those pairwise
opinions into one complete order.

### The three decoders

| Decoder | Idea | Use it when |
| --- | --- | --- |
| `borda` | Give each sentence an average "before the others" score, then sort. | Default; robust and fast. |
| `clusters` | Group similar sentences with TF-IDF + KMeans, order the groups, then order inside each group. | You want the assignment's clustering idea. |
| `beam` | Search for the highest-scoring chain of sentences. | You want a single global chain, but it is slower. |

The evaluation also reports a **TF-IDF-only ablation**: ordering without the
pair model. It is a baseline we must beat.


In [ ]:
# Stage 2: sentence ordering.
!python -m src.ordering.train \
    --epochs 2 \
    --batch-size 8 \
    --learning-rate 2e-4 \
    --memory-gb 1.5


### 5.1 Read the ordering results

Ordering is harder than punctuation. The model sees one pair at a time, and the
final article order is assembled from many noisy pairwise decisions.

- **pair accuracy:** how often the model puts a pair in the right order.
  Anything near `0.5` is guessing; we want clearly above `0.5`.
- **macro F1:** averages the score for both classes, so it does not hide a
  model that always predicts the majority class.
- **Kendall's tau:** computed later on complete shuffled articles. `0` means no
  relationship with the true order, `1` means perfect order.

The local verified run reached **0.6543 pair accuracy** and **0.2007 Borda
Kendall's tau** with a small subset. The full Colab recipe trains on 40,000
pairs and should be stronger.


In [ ]:
order_summary = json.loads((ORDER_MODEL_DIR / "training_summary.json").read_text())
print("Trainable parameters:", order_summary.get("trainable_parameters"))
print("Validation:", json.dumps(order_summary["validation_metrics"], indent=2))
print("Test:", json.dumps(order_summary["test_metrics"], indent=2))
print("Peak memory:", order_summary["memory_at_end"])


## 6. Evaluation: how do we know it works?

Training loss tells us how well the model fits the training data. It does not
tell us whether the model is useful on unseen text. Evaluation does.

### Metrics in one minute

| Metric | Question it answers |
| --- | --- |
| **Accuracy** | Out of all decisions, how many were correct? |
| **Precision** | When the model says "punctuation here", how often is it right? |
| **Recall** | Out of all real punctuation marks, how many did it find? |
| **F1** | A balance between precision and recall. Higher is better. |
| **Boundary F1** | F1 for the yes/no question "is this a sentence boundary?" |
| **Macro F1** | Average F1 across classes, treating each class equally. |
| **Kendall's tau** | How similar the recovered order is to the true order. |
| **BLEU** | Translation score based on matching word sequences. |
| **chrF** | Translation score based on character overlap. |

The next command evaluates all three stages. Translation uses 200 public IITB
Hindi-English test sentences. It is skipped gracefully if that dataset is
unavailable.


In [ ]:
# Evaluation is inference only: no training happens here.
!python -m src.evaluate \
    --max-punct-examples 1200 \
    --max-order-pairs 3000 \
    --max-order-articles 30 \
    --max-translation-examples 200


### 6.1 Read the evaluation report

The raw JSON is long, so the next cell prints only the numbers that matter and
one translation example.

How to interpret them:

- punctuation accuracy is usually high because `none` is common; boundary F1 is
  the meaningful punctuation score,
- ordering pair accuracy must beat `0.5`, and Borda tau should beat the
  TF-IDF-only ablation,
- translation BLEU/chrF are comparative scores, not percentages; exact wording
  can vary with the model version.


In [ ]:
evaluation = json.loads(Path("reports/evaluation.json").read_text())

punct = evaluation["punctuation"]
ordering = evaluation["ordering"]
translation = evaluation["translation"]

print("PUNCTUATION")
print("  examples:", punct["examples"], "| words:", punct["words"])
print("  accuracy:", punct["accuracy"])
print("  boundary precision / recall / F1:",
      punct["boundary"]["precision"],
      punct["boundary"]["recall"],
      punct["boundary"]["f1"])
print("  per-class F1:", punct["per_class_f1"])

print("\nORDERING")
print("  pairs:", ordering["pairs"]["pairs"])
print("  pair accuracy:", ordering["pairs"]["accuracy"])
print("  pair macro F1:", ordering["pairs"]["macro_f1"])
for name, stats in ordering["kendall_tau"].items():
    print(f"  {name:16s} mean tau: {stats['mean_tau']}")

print("\nTRANSLATION")
print("  status:", translation["status"])
if translation["status"] == "ok":
    print("  examples:", translation["examples"])
    print("  BLEU / chrF:", translation["bleu"], "/", translation["chrf"])
    sample = translation["samples"][0]
    print("  example Hindi:  ", sample["hindi"])
    print("  reference:      ", sample["reference"])
    print("  model output:   ", sample["hypothesis"])
else:
    print("  reason:", translation.get("reason"))


### 6.2 Check the results before running the pipeline

The verifier checks that the adapters exist, the report is complete, and the
scores are above conservative minimums. It deliberately runs **before** the
pipeline so a broken model fails in seconds instead of after a long inference
run.

If translation is `skipped` because the public IITB dataset is temporarily
unavailable, run the verifier manually with `--allow-skipped-translation`. The
pipeline itself still needs the gated IndicTrans2 checkpoint.


In [ ]:
!python scripts/verify_colab_run.py --skip-outputs --skip-model-smoke


## 7. Run the full pipeline on the assignment files

Now we connect the three stages. For each `data/raw/sample-N.txt`, the pipeline:

1. removes `NULL` tokens,
2. restores punctuation and sentence boundaries,
3. orders the sentences,
4. translates them to English,
5. groups every five sentences,
6. writes four text files.

Models are loaded **one at a time**. After a stage finishes, its model is
deleted and GPU memory is freed. That is how the pipeline stays under the 1.5 GB
cap even though it uses three models.

The default ordering strategy is `borda`.


In [ ]:
!python -m src.pipeline --strategy borda --memory-gb 1.5


### 7.1 Inspect the output files

For every sample you should see:

| File | What it contains |
| --- | --- |
| `sample-N.sentences_hi.txt` | Punctuated Hindi sentences, one per line, in recovered order |
| `sample-N.groups_hi.txt` | The same Hindi sentences grouped five at a time |
| `sample-N.sentences_en.txt` | English translations, line by line |
| `sample-N.groups_en.txt` | English translations grouped five at a time |

The critical property is **alignment**: sentence `i` in the Hindi file must
correspond to sentence `i` in the English file, and the group files must contain
exactly the same sentences in the same order.


In [ ]:
for name in ["sample-1.groups_hi.txt", "sample-1.groups_en.txt"]:
    path = Path("outputs", name)
    print("=" * 100)
    print(name, f"({len(path.read_text(encoding='utf-8').splitlines())} groups)")
    for index, line in enumerate(path.read_text(encoding="utf-8").splitlines()[:4]):
        print(f"[group {index + 1}] {line}")

print("\nAlignment check:")
for stem in [f"sample-{index}" for index in range(1, 8)]:
    hi = Path(f"outputs/{stem}.sentences_hi.txt").read_text(encoding="utf-8").splitlines()
    en = Path(f"outputs/{stem}.sentences_en.txt").read_text(encoding="utf-8").splitlines()
    ghi = Path(f"outputs/{stem}.groups_hi.txt").read_text(encoding="utf-8").splitlines()
    gen = Path(f"outputs/{stem}.groups_en.txt").read_text(encoding="utf-8").splitlines()
    print(f"  {stem}: {len(hi)} hi / {len(en)} en sentences, "
          f"{len(ghi)} hi / {len(gen)} en groups, aligned={len(hi) == len(en)}")


### 7.2 What good output looks like

- Every line is non-empty.
- Hindi lines contain Devanagari characters and punctuation.
- English lines contain readable English words.
- Hindi and English sentence counts match.
- Group counts match `ceil(sentences / 5)`.
- No line contains `NULL`.
- The last group may have fewer than five sentences; that is expected.

The final self-check below verifies all of this automatically.


## 8. Memory and timing

The assignment caps GPU memory at **1.5 GB**. The pipeline reports how long each
stage took. Ordering and translation are usually the slowest because they make
many model calls.

`run_summary.json` also records, for every input file:

- words read,
- `NULL` tokens removed,
- sentences produced,
- groups produced.

If you change the ordering strategy, compare the timings and the tau scores.
There is a real trade-off between quality and speed.


In [ ]:
run_summary = json.loads(Path("reports/run_summary.json").read_text())
print("Ordering strategy:", run_summary["ordering_strategy"])
print("Timings (seconds):", json.dumps(run_summary["timings_seconds"], indent=2))
print("Memory limit (GB):", run_summary["memory_limit_gb"])
print()
for stem, stats in run_summary["files"].items():
    print(stem, stats)


## 9. Final end-to-end self-check

This is the last gate. It re-checks the data and reports, runs a smoke test on
both adapters, and verifies that all seven inputs produced four aligned,
non-empty files. If this passes, the run is complete.


In [ ]:
!python scripts/verify_colab_run.py


## 10. Save your results

Colab deletes its files when the runtime ends. The next cell creates a zip of
`outputs/` and copies `reports/` to `/content/`. Download both before closing
the tab.

What to keep for submission:

- the four output files for every sample,
- `reports/data_prep_summary.json`,
- `reports/evaluation.json`,
- `reports/run_summary.json`,
- the two `training_summary.json` files from `models/`,
- this notebook, with its outputs visible.


In [ ]:
import shutil

shutil.make_archive("/content/hindi_assignment_outputs", "zip", "outputs")
shutil.copytree("reports", "/content/hindi_assignment_reports", dirs_exist_ok=True)
print("Saved /content/hindi_assignment_outputs.zip")
print("Saved /content/hindi_assignment_reports/")


## 11. Troubleshooting

| Problem | What to do |
| --- | --- |
| `No GPU detected` | Runtime > Change runtime type > T4 GPU, then rerun from the top. |
| `401` or gated repo for IndicTrans2 | Accept the licence on Hugging Face and run `huggingface-cli login`. |
| `IndicTransToolkit` build error | Re-run `bash scripts/colab_setup.sh`; it falls back to a source build. |
| Import errors after install | Runtime > Restart runtime, then run from the top. The install cell is safe to repeat. |
| CUDA out of memory | Restart runtime, keep `--memory-gb 1.5`, reduce `--batch-size` to 4. |
| Self-check failed | Read the printed failure list; it names the missing file or threshold. |
| Data download is slow | Use the smaller run from the README: fewer articles and training samples. |


### Where to read the code next

1. `src/constants.py` - all settings in one place.
2. `src/pipeline.py` - the three stages connected.
3. `src/punctuation/` - dataset, training, inference.
4. `src/ordering/` - pairs, training, decode strategies.
5. `src/translation/indictrans.py` - the translator wrapper.
6. `src/data_prep.py` and `src/evaluate.py` - how data and metrics are built.

### Exercises

- Run the pipeline with `--strategy clusters` and compare its Kendall tau with
  `borda` in `reports/evaluation.json`.
- Change `--epochs` from 3 to 1 and observe the trade-off between time and
  boundary F1.
- Add a new `.txt` file to `data/raw/` and rerun only the pipeline.
- Read one `training_summary.json` and explain which class has the lowest F1 and
  why.


## Glossary

| Term | Meaning |
| --- | --- |
| **Adapter** | The saved LoRA weights in `models/punctuation/adapter/` or `models/ordering/adapter/`. |
| **Batch** | A small group of examples processed together. |
| **BLEU** | A translation score based on matching word sequences. Higher is better. |
| **Borda** | A voting-style decoder: average each sentence's "before others" probability, then sort. |
| **chrF** | A translation score based on character overlap. Higher is better. |
| **Epoch** | One complete pass over the training data. |
| **F1** | A balance between precision and recall. |
| **Inference** | Using a trained model to make predictions. |
| **Kendall's tau** | A score from -1 to 1 measuring how similar two orderings are. |
| **LoRA** | A small set of trainable weights added to a frozen model. |
| **Loss** | A number the model tries to make smaller during training. |
| **mBERT** | The multilingual BERT model used for punctuation and ordering. |
| **NULL** | A placeholder token in the assignment input; it is removed before processing. |
| **Seed** | A number that makes random choices repeatable. |
| **Softmax** | Turns raw model scores into probabilities that sum to 1. |
| **Subword** | A piece of a word produced by the tokenizer. |
| **Token** | The smallest unit of text the model reads. |
